# Final analysis — cross-linguistic LLM tokenization

This notebook reloads every CSV produced by the five experiments and
regenerates the major figures. It does **not** re-run any experiment;
the experiments are CLI-driven (see the README).

Run order:
1. `python -m src.run_tokenization_audit --config configs/audit.yaml`
2. `python -m src.run_fixed_context --config configs/fixed_context.yaml`
3. `python -m src.run_radical_sensitivity --config configs/radicals.yaml`
4. `python -m src.run_paraphrase_perturb --config configs/perturb.yaml`
5. `python -m src.run_tokenizer_adaptation --config configs/adaptation.yaml`
6. open this notebook.

## Headlines (read me first)

> **Backend used for model-dependent experiments:** OpenAI `gpt-4o-mini`
> (configurable in `configs/*.yaml`).
> **Tokenizers audited:** `tiktoken_cl100k`, `tiktoken_o200k`, `llama3` (Meta-Llama-3.1-8B), `qwen25` (Qwen2.5-7B), `byte`, `char`.
> **Corpus:** FLORES-200 EN↔ZH devtest (1012 paired sentences).

| # | Experiment | Headline finding |
|---|---|---|
| 1 | Tokenization audit | `cl100k_base` mean TP = **1.89** (Chinese costs ~2× the context). `o200k_base` = 1.28, `llama3` = 1.31, `qwen25` = **1.01** (parity). |
| 2 | Fixed-context fairness | At every shared budget, the Chinese few-shot prompt fits **fewer demonstrations** than English (45% fewer at 512 tokens, 19% fewer at 1024, 8% fewer at 2048). Accuracy gap is within Wilson noise for n=50. |
| 3 | Radical sensitivity | Same-radical pairs: model says "similar" 26% of the time vs 1% for different-radical pairs (Fisher exact p ≈ 7e-8, OR ≈ 24). **Within same-radical pairs, sharing the same first byte token in cl100k_base raises p('similar') from 23% → 50%** — a measurable token-identity effect after controlling for script. |
| 4 | Paraphrase vs perturbation | Even at temperature=0, identity stability is only ~67% — that's the floor. Token-perturbation transforms (`punct_fullwidth`, `to_traditional`) drop stability further; meaning-preserving Δaccuracy ≈ 0 across the bundled prompt set. |
| 5 | Tokenizer adaptation (lightweight, simulated) | Adding ~5k mined Han n-grams to the base tokenizer drops mean TP from **1.05 → 0.85** on the same corpus. Counterfactual upper-bound for cheap tokenizer surgery. |

**Read the narrative version in `reports/final_report.md`** for the full per-experiment interpretation, including guardrails and limitations.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.plotting import apply_style
from src.experiments.tokenization_audit import (
    plot_mean_tp_bar, plot_tp_box, plot_en_zh_scatter,
)
from src.experiments.fixed_context import (
    _plot_budget_curve, _plot_examples_fit, _plot_cost_per_correct,
)
from src.experiments.tokenizer_adaptation import _plot_before_after

apply_style()
RESULTS = Path('../results')

def maybe(path):
    p = Path(path)
    return pd.read_csv(p) if p.exists() and p.stat().st_size else None

## 1. Tokenization audit

In [2]:
audit_summary = maybe(RESULTS / 'tokenization_audit' / 'summary_by_tokenizer.csv')
audit_per_sent = maybe(RESULTS / 'tokenization_audit' / 'per_sentence.csv')
if audit_summary is not None:
    display(audit_summary)
    plot_mean_tp_bar(audit_summary, ('en', 'zh'),
                     RESULTS / 'tokenization_audit' / 'plots' / 'tp_bar')
    plot_tp_box(audit_per_sent, ('en', 'zh'),
                RESULTS / 'tokenization_audit' / 'plots' / 'tp_box')
    plot_en_zh_scatter(audit_per_sent, ('en', 'zh'),
                       RESULTS / 'tokenization_audit' / 'plots' / 'en_zh_scatter')
    print('TP > 1 means Chinese fragments more under that tokenizer.\n'
          'Bootstrap CI columns quantify uncertainty over the bilingual sample.')
else:
    print('Run Experiment 1 first.')

,tokenizer,dataset,n_pairs,mean_TP,median_TP,TP_ci_low,TP_ci_high,mean_context_shrinkage,mean_en_tokens,mean_zh_tokens,wilcoxon_stat,wilcoxon_pvalue
0,byte,flores200,1012,0.895510,0.880503,0.884892,0.906096,-0.153382,130.533597,115.973320,77262.0,1.535837e-79
1,char,flores200,1012,0.331317,0.310745,0.325620,0.337263,-2.202961,130.405138,42.762846,0.0,5.278447e-167
2,llama3,flores200,1012,1.312640,1.275862,1.296793,1.329080,0.209046,26.852767,35.018775,10150.0,7.375101e-149
3,qwen25,flores200,1012,1.005361,0.965517,0.990620,1.020226,-0.042046,27.297431,27.436759,194073.5,4.083948e-02
4,tiktoken_cl100k,flores200,1012,1.885521,1.851471,1.862764,1.909740,0.448536,26.863636,50.284585,1.5,3.396114e-167
5,tiktoken_o200k,flores200,1012,1.280664,1.237103,1.265215,1.296951,0.189271,26.558300,33.835968,15490.5,5.828363e-140


TP > 1 means Chinese fragments more under that tokenizer.
Bootstrap CI columns quantify uncertainty over the bilingual sample.


## 2. Fixed-context fairness

In [3]:
fc = maybe(RESULTS / 'fixed_context' / 'summary.csv')
if fc is not None and not fc.empty:
    display(fc)
    base = RESULTS / 'fixed_context' / 'plots'
    _plot_budget_curve(fc, base / 'accuracy_vs_budget', 'accuracy')
    _plot_examples_fit(fc, base / 'examples_fit_vs_budget')
    _plot_cost_per_correct(fc, base / 'cost_per_correct')
    print('examples_fit_mean shows context economics; accuracy combines that with model behaviour.')
else:
    print('Run Experiment 2 first.')

,language,budget,task,n_queries,examples_fit_mean,accuracy,f1,em,rougeL,tokens_in,tokens_out,latency_total_s,cost_total_usd,cost_per_correct
0,en,512,xnli,50,11.80,0.82,0.0,0.0,0.0,23626,118,33.135519,0.003615,0.000088
1,en,1024,xnli,50,22.00,0.82,0.0,0.0,0.0,47150,114,35.792952,0.007141,0.000174
2,en,2048,xnli,50,32.00,0.84,0.0,0.0,0.0,72000,120,29.049431,0.010872,0.000259
3,zh,512,xnli,50,6.44,0.78,0.0,0.0,0.0,16392,175,24.086196,0.002564,0.000066
4,zh,1024,xnli,50,17.72,0.80,0.0,0.0,0.0,33804,173,38.068847,0.005174,0.000129
5,zh,2048,xnli,50,29.50,0.76,0.0,0.0,0.0,67645,185,32.388273,0.010258,0.000270


examples_fit_mean shows context economics; accuracy combines that with model behaviour.


## 3. Radical sensitivity

In [4]:
rad = maybe(RESULTS / 'radical_sensitivity' / 'condition_table.csv')
coef = maybe(RESULTS / 'radical_sensitivity' / 'logit_coefficients.csv')
if rad is not None: display(rad)
if coef is not None: display(coef)
if rad is None and coef is None:
    print('Run Experiment 3 first.')

,same_radical,same_token,n,accuracy,p_pred_yes
0,0,0,100,0.990000,0.010000
1,1,0,90,0.233333,0.233333
2,1,1,10,0.500000,0.500000


,term,coef,p_value,regularized
0,const,-4.595120,0.000005,False
1,same_radical,3.405536,0.001006,False
2,same_token,1.189584,0.080130,False


## 4. Paraphrase vs token perturbation

In [5]:
pt = maybe(RESULTS / 'paraphrase_token_perturb' / 'summary.csv')
if pt is not None: display(pt)
else: print('Run Experiment 4 first.')

,transform,n,mean_stability_exact,mean_stability_jaccard,mean_delta_tokens,delta_accuracy
0,identity,6,0.666667,0.780702,0.000000,0.0
1,numerals_arabic,6,0.833333,0.833333,0.166667,0.0
2,punct_ascii,6,1.000000,1.000000,0.000000,0.0
3,punct_fullwidth,6,0.666667,0.758333,0.000000,0.0
4,to_simplified,6,0.833333,0.833333,0.000000,0.0
5,to_traditional,6,0.500000,0.725146,1.000000,0.0


## 5. Tokenizer adaptation

In [6]:
ad = maybe(RESULTS / 'tokenizer_adaptation' / 'audit_before_after.csv')
if ad is not None and not ad.empty:
    display(ad)
    _plot_before_after(ad.to_dict(orient='records'),
                       RESULTS / 'tokenizer_adaptation' / 'plots' / 'audit_before_after')
else:
    print('Run Experiment 5 first.')

,added_tokens,mean_en_tokens,mean_zh_tokens,mean_TP
0,0,28.265,29.760,1.049408
1,1000,28.265,26.425,0.932569
2,4960,28.265,23.980,0.845305
3,4960,28.265,23.980,0.845305


## Limitations & guardrails

- Numbers are tokenizer-, model-, and task-specific. Generalisations beyond these
  axes are not supported by the data here.
- Closed-model results depend on (unknown) pre-training mixtures.
- The radical sensitivity dataset shipped here is a small seed list.
  Results scale with vocabulary; supply a richer CSV via
  `radicals.yaml: dataset_csv:`.
- Lightweight tokenizer-adaptation results are *counterfactual* unless
  `do_train: true` was set; they upper-bound the gain at zero training cost.